# Reddit_scrapy.ipynb

**Author:** Zane Zhang

**Date:** 2025/06/12 13:53

**Description:** 

This script fetches data using PRAW from the Reddit and saves it to a CSV file.
    
First, we define a search query with keywords related to depression.  

Second, we define a function to search and scrape data based on our query, containing the post/submission and its comments. In this step, a whitelist of subreddit is applied. What should be paid attention to is that although we confine the results within related keywords, Reddit still conducts a loose relevance search due to its system, which means the results could contain irrelevant information in the posts.

To achieve our goal, we fetch posts and comments respectively due to avoid rate limitations.

Attentively, PRAW has been emphasized by official documents that it is not thread safe, thus every thread needs an instance.

In [ ]:
import praw
import pandas as pd
import time
import traceback
import random
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import cpu_count

# Client information
client_id = "fdIC_cvzN3qisYXCujsUUg"
client_secret = "MKhf5qMY-aGwcP-rmWIruLDXPGMGkw"
user_agent = "windows:DepressionMIsinformationBot (by u/Glass_Farmer2453)"

search_keywords = [
    # core depression terms
    "depression",
    "depressed",
    "depressive disorder",
    "major depressive disorder",
    "mdd",   # abbreviation
    
    # treatment
    "antidepressants",
    "antidepressant medication",
    "ssri",
    "snri",
    "CBT",
    "DBT",
    "psychotherapy",
    "therapy for depression",
    "diagnosed with depression",
    "depression diagnosis",
    "depression medication",
]

whitelist_subreddits = [
    # Science, medicine, and health info communities
    "science",
    "medicine",
    "askscience",
    "AskDocs",
    "Health",
    
    # Public discussion and Q&A subreddits
    "AskReddit",
    "TooAfraidToAsk",
    "NoStupidQuestions",
    "explainlikeimfive",

    # News and societal discourse
    "news",
    "worldnews",
    "changemyview",

    # General well-being and advice
    "mentalhealth",
    "selfhelp",
    "therapy",
]

In [4]:
# define function to scrape data from subreddit
def scrape_posts(subreddit, keyword):
    '''
    PRAW is unsafe for pool and it should have an instance for every thread.
    Note: even if args are same across threads, if you set a instance within the thread, it will be a new instance
    '''
    # set a instance
    reddit = praw.Reddit(
        client_id = client_id,
        client_secret = client_secret,
        user_agent = user_agent
    )
    
    local_data = []     # define a dataset in current thread
    local_seen_ids = set()

    # search and scape
    # print(f"[r/{subreddit}] Searching keyword: {keyword}")   
    try:
        for post in reddit.subreddit(subreddit).search(query=keyword, time_filter="year", sort="relevance", limit=500):
            if post.id in local_seen_ids:
                continue    # if post id has exist in set, skip this post
            local_seen_ids.add(post.id)
            if not (post.is_self and post.selftext.strip()):  
                continue    # if not only non-empty text posts, skip
            try:
                local_data.append({
                    'Subreddit': post.subreddit.display_name,
                    'Type': 'Post',
                    'Post_id': post.id,
                    'Title': post.title,
                    'Author': post.author.name if post.author else 'Unknown',
                    'Timestamp': pd.to_datetime(post.created_utc, unit='s'),
                    'Text': post.selftext,
                    'Score': post.score,
                    'Total_comments': post.num_comments,
                    'Post_url': post.url,
                    'Keyword_matched': keyword
                })
                time.sleep(1)       
            except Exception as e:
                print(f"Error while processing a post: {e}")
                traceback.print_exc()
    except Exception as e:
        print(f"Error during search in r/{subreddit} for keyword {keyword}: {e}")
        traceback.print_exc()

    # print(f"[r/{subreddit}] Finished keyword: {keyword}")
    return local_data

In [5]:
# scrape comments for every post
def scrape_comments(post_id):
    '''fetch comments by post id'''
    reddit = praw.Reddit(
        client_id = client_id,
        client_secret = client_secret,
        user_agent = user_agent
    )

    comments = []
    try:
        post = reddit.submission(id=post_id)
        post.comments.replace_more(limit=None)     # unfold all comments 
        for comment in post.comments.list():
            comments.append({
                'Subreddit': post.subreddit.display_name,
                'Type': 'Comment',
                'Post_id': post.id,     # associate with post 
                'Title': post.title,
                'Author': comment.author.name if comment.author else 'Unknown',
                'Timestamp': pd.to_datetime(comment.created_utc, unit='s'),
                'Text': comment.body,
                'Score': comment.score,
                'Total_comments': 0,    # comments don't have this attribute
                'Post_url': None,    # comments don't have this attribute
                'Keyword_matched': None
            })
            time.sleep(1)
    except Exception as e:
        print(f"Error in scrape_comments for post {post_id}: {e}")
        traceback.print_exc()
    return comments 

In [6]:
# define a function to fix subreddit
def search_in_subreddit(subreddit):
    '''input a subreddit and search keywords within it'''
    print(f"[r/{subreddit}] STARTED")
    subreddit_data = []
    for keyword in search_keywords:
        results = scrape_posts(subreddit, keyword)
        subreddit_data.extend(results)
        time.sleep(random.uniform(2, 4))
    print(f"[r/{subreddit}] FINISHED")
    time.sleep(random.uniform(10, 20))  # to avoid accumulated speed
    return subreddit_data

In [ ]:
posts = []
with ThreadPoolExecutor(cpu_count()) as executor: 
    results = list(executor.map(search_in_subreddit, whitelist_subreddits))
    for r in results:
        posts.extend(r)

posts_data = pd.DataFrame(posts)     # create pandas DataFrame for posts and comments
posts_data = posts_data.drop_duplicates(subset=['Post_id', 'Type', 'Text'])     # drop duplicates across different threads 
posts_data.to_csv('data/posts.csv', encoding="utf-8", index=False)  # Save DataFrame

print(f"Number of posts: {len(posts_data)}")
print("Scraping posts completed")

[r/science] STARTED
[r/medicine] STARTED
[r/askscience] STARTED
[r/AskDocs] STARTED
[r/Health] STARTED
[r/AskReddit] STARTED
[r/TooAfraidToAsk] STARTED
[r/NoStupidQuestions] STARTED
[r/explainlikeimfive] STARTED
[r/news] STARTED
[r/worldnews] STARTED
[r/changemyview] STARTED
[r/mentalhealth] STARTED
[r/selfhelp] STARTED
[r/therapy] STARTED
[r/news] FINISHED
[r/worldnews] FINISHED
[r/Health] FINISHED
[r/askscience] FINISHED
[r/science] FINISHED
[r/AskReddit] FINISHED
[r/medicine] FINISHED
[r/explainlikeimfive] FINISHED
[r/changemyview] FINISHED
[r/TooAfraidToAsk] FINISHED
[r/selfhelp] FINISHED
[r/NoStupidQuestions] FINISHED
[r/therapy] FINISHED
[r/AskDocs] FINISHED
[r/mentalhealth] FINISHED


In [ ]:
posts_data = pd.read_csv('data/posts.csv')
comments = []
post_ids = posts_data['Post_id'].tolist()
with ThreadPoolExecutor(max_workers=4) as t:
    comment = list(t.map(scrape_comments, post_ids))
    for c in comment:
        comments.extend(c)

comments = pd.DataFrame(comments)
comments.to_csv('data/comments.csv', encoding='utf-8', index=False)

print("Scraping comments completed")

Scraping comments completed


In [8]:
print(f"Number of comments: {len(comments)}")
print(f"Number of posts: {len(posts_data)}")

Number of comments: 129221
Number of posts: 6688
